In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DataCoSupplyChainDataset.csv", encoding="cp1252")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Dataset loaded successfully!
Rows: 180519
Columns: 53


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [3]:
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'], errors='coerce'
)

df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)'], errors='coerce'
)

df['Days for shipping (real)'] = pd.to_numeric(
    df['Days for shipping (real)'], errors='coerce'
)

df['Days for shipment (scheduled)'] = pd.to_numeric(
    df['Days for shipment (scheduled)'], errors='coerce'
)

print("Data preparation completed!")

Data preparation completed!


In [4]:
baseline = df.copy()

baseline_result = pd.DataFrame({
    'Metric': [
        'Total Records',
        'Average Delivery Duration',
        'Median Delivery Duration',
        'Minimum Duration',
        'Maximum Duration',
        'Late Deliveries',
        'On-Time Deliveries'
    ],
    'Value': [
        len(baseline),
        baseline['Days for shipping (real)'].mean(),
        baseline['Days for shipping (real)'].median(),
        baseline['Days for shipping (real)'].min(),
        baseline['Days for shipping (real)'].max(),
        (baseline['Days for shipping (real)'] >
         baseline['Days for shipment (scheduled)']).sum(),
        (baseline['Days for shipping (real)'] ==
         baseline['Days for shipment (scheduled)']).sum()
    ]
})

display(baseline_result)

,Metric,Value
0,Total Records,180519.000000
1,Average Delivery Duration,3.497654
2,Median Delivery Duration,3.000000
3,Minimum Duration,0.000000
4,Maximum Duration,6.000000
5,Late Deliveries,103400.000000
6,On-Time Deliveries,33753.000000


In [5]:
missing_count = df[
    ['order date (DateOrders)', 'shipping date (DateOrders)']
].isnull().sum()

valid_timestamp_df = df.dropna(
    subset=[
        'order date (DateOrders)',
        'shipping date (DateOrders)'
    ]
)

missing_timestamp_result = pd.DataFrame({
    'Metric': [
        'Total Records',
        'Missing Order Timestamps',
        'Missing Shipping Timestamps',
        'Records With Valid Timestamps',
        'Average Delivery Duration After Exclusion'
    ],
    'Value': [
        len(df),
        missing_count.iloc[0],
        missing_count.iloc[1],
        len(valid_timestamp_df),
        valid_timestamp_df['Days for shipping (real)'].mean()
    ]
})

display(missing_timestamp_result)

,Metric,Value
0,Total Records,180519.000000
1,Missing Order Timestamps,0.000000
2,Missing Shipping Timestamps,0.000000
3,Records With Valid Timestamps,180519.000000
4,Average Delivery Duration After Exclusion,3.497654


In [6]:
duration = df['Days for shipping (real)'].dropna()

Q1 = duration.quantile(0.25)
Q3 = duration.quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

unusual_records = df[
    (df['Days for shipping (real)'] < lower_limit) |
    (df['Days for shipping (real)'] > upper_limit)
]

normal_records = df[
    (df['Days for shipping (real)'] >= lower_limit) &
    (df['Days for shipping (real)'] <= upper_limit)
]

unusual_duration_result = pd.DataFrame({
    'Metric': [
        'Q1',
        'Q3',
        'IQR',
        'Lower Limit',
        'Upper Limit',
        'Unusual Records',
        'Normal Records',
        'Average Duration Before Exclusion',
        'Average Duration After Exclusion'
    ],
    'Value': [
        Q1, Q3, IQR, lower_limit, upper_limit,
        len(unusual_records),
        len(normal_records),
        duration.mean(),
        normal_records['Days for shipping (real)'].mean()
    ]
})

display(unusual_duration_result)

,Metric,Value
0,Q1,2.000000
1,Q3,5.000000
2,IQR,3.000000
3,Lower Limit,-2.500000
4,Upper Limit,9.500000
5,Unusual Records,0.000000
6,Normal Records,180519.000000
7,Average Duration Before Exclusion,3.497654
8,Average Duration After Exclusion,3.497654


In [7]:
missing_timestamps = df[
    df['order date (DateOrders)'].isna() |
    df['shipping date (DateOrders)'].isna()
]

negative_duration = df[
    df['Days for shipping (real)'] < 0
]

duplicate_records = df[df.duplicated()]

documented_exclusions = pd.DataFrame({
    'Exclusion Reason': [
        'Missing Order/Shipping Timestamps',
        'Negative Delivery Duration',
        'Exact Duplicate Records'
    ],
    'Excluded Records': [
        len(missing_timestamps),
        len(negative_duration),
        len(duplicate_records)
    ],
    'Reason': [
        'Required timestamps are unavailable',
        'Negative delivery duration is invalid',
        'Duplicate records may affect analysis'
    ]
})

display(documented_exclusions)

,Exclusion Reason,Excluded Records,Reason
0,Missing Order/Shipping Timestamps,0,Required timestamps are unavailable
1,Negative Delivery Duration,0,Negative delivery duration is invalid
2,Exact Duplicate Records,0,Duplicate records may affect analysis


In [8]:
clean_df = df.dropna(
    subset=[
        'order date (DateOrders)',
        'shipping date (DateOrders)'
    ]
).copy()

clean_df = clean_df[
    clean_df['Days for shipping (real)'] >= 0
]

clean_df = clean_df.drop_duplicates()

sensitivity_summary = pd.DataFrame({
    'Metric': [
        'Total Records',
        'Average Delivery Duration',
        'Median Delivery Duration',
        'Minimum Delivery Duration',
        'Maximum Delivery Duration'
    ],
    'Baseline': [
        len(df),
        df['Days for shipping (real)'].mean(),
        df['Days for shipping (real)'].median(),
        df['Days for shipping (real)'].min(),
        df['Days for shipping (real)'].max()
    ],
    'After Exclusions': [
        len(clean_df),
        clean_df['Days for shipping (real)'].mean(),
        clean_df['Days for shipping (real)'].median(),
        clean_df['Days for shipping (real)'].min(),
        clean_df['Days for shipping (real)'].max()
    ]
})

sensitivity_summary['Difference'] = (
    sensitivity_summary['After Exclusions'] -
    sensitivity_summary['Baseline']
)

display(sensitivity_summary)

,Metric,Baseline,After Exclusions,Difference
0,Total Records,180519.000000,180519.000000,0.0
1,Average Delivery Duration,3.497654,3.497654,0.0
2,Median Delivery Duration,3.000000,3.000000,0.0
3,Minimum Delivery Duration,0.000000,0.000000,0.0
4,Maximum Delivery Duration,6.000000,6.000000,0.0


In [9]:
output_file = "11_Robustness_Checks.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

    baseline_result.to_excel(
        writer, sheet_name='Baseline_Result', index=False
    )

    missing_timestamp_result.to_excel(
        writer, sheet_name='Missing_Timestamps', index=False
    )

    unusual_duration_result.to_excel(
        writer, sheet_name='Unusual_Durations', index=False
    )

    documented_exclusions.to_excel(
        writer, sheet_name='Documented_Exclusions', index=False
    )

    sensitivity_summary.to_excel(
        writer, sheet_name='Sensitivity_Summary', index=False
    )

print("Excel workbook generated successfully!")
print("File Name:", output_file)

Excel workbook generated successfully!
File Name: 11_Robustness_Checks.xlsx
